safe-route-app/
├─ docker-compose.yml
├─ .env.example
├─ db/
│  └─ init.sql
└─ api/
   ├─ Dockerfile
   ├─ requirements.txt
   └─ main.py

In [ ]:
#docker-compose.yml
version: "3.9"

version: "3.9"

services:
  db:
    image: postgis/postgis:16-3.4
    container_name: safe_db
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: safe_map
    ports: ["5432:5432"]
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./db/init.sql:/docker-entrypoint-initdb.d/00-init.sql:ro
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres -d safe_map"]
      interval: 5s
      timeout: 5s
      retries: 10

  api:
    build: ./api
    container_name: safe_api
    env_file: .env
    environment:
      DATABASE_URL: ${DATABASE_URL:-postgresql+psycopg://postgres:postgres@db:5432/safe_map}
    depends_on:
      db:
        condition: service_healthy
    ports: ["8000:8000"]
    command: uvicorn main:app --host 0.0.0.0 --port 8000 --reload

volumes:
  pgdata:


In [ ]:
#.env.example
DATABASE_URL=postgresql+psycopg://postgres:postgres@db:5432/safe_map


In [ ]:
#api/Dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py ./

EXPOSE 8000


In [ ]:
#api/requirements.txt
fastapi==0.115.5
uvicorn==0.30.6
pydantic==2.9.2
SQLAlchemy==2.0.35
psycopg==3.2.3
python-dotenv==1.0.1
